In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.inspection import permutation_importance

In [ ]:
# -------------------------
# 0) Load / define your data
# -------------------------
# df = pd.read_csv("your_dataset.csv")

FEATURES = [
    "CIN_ml", "CAPE_ml",
    "LCL_ml_m", "PBLH_m", "LCL_minus_PBLH_m",
    "RH700", "RH500", "midlevel_dryness",
    "lr_700_500", "shear_0_6"
]
TARGET = "y"

# Basic hygiene: drop rows with missing required columns
df_model = df.dropna(subset=FEATURES + [TARGET]).copy()

X = df_model[FEATURES]
y = df_model[TARGET].astype(int)

# -------------------------
# 1) Train/test split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# -------------------------
# 2) Fit gradient boosting
# -------------------------
gb = HistGradientBoostingClassifier(
    max_depth=3,          # shallow trees = more interpretable, less overfit
    learning_rate=0.07,
    max_iter=400,
    min_samples_leaf=25,
    l2_regularization=0.2,
    random_state=42
)
gb.fit(X_train, y_train)

# -------------------------
# 3) Evaluate (rare-event friendly metrics)
# -------------------------
proba = gb.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

roc = roc_auc_score(y_test, proba)
ap  = average_precision_score(y_test, proba)

print(f"ROC AUC: {roc:.3f}")
print(f"PR AUC : {ap:.3f}")
print(classification_report(y_test, pred, digits=3))

# -------------------------
# 4) “Key variables” via permutation importance
# -------------------------
perm = permutation_importance(
    gb, X_test, y_test,
    n_repeats=20,
    random_state=42,
    scoring="roc_auc"
)

imp = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print("\nTop variables (permutation importance: ROC AUC drop):")
print(imp.head(15).to_string(index=False))
